# Sci-Fi TF-IDF Clustering

Fetches full Wikipedia article text for every page in an extract table,
builds a TF-IDF matrix, reduces to 2-D with UMAP, and clusters with KMeans.

**Kernel**: select **"Python 3.12 (RAPIDS GPU)"** in Jupyter to get full GPU
acceleration (cuML UMAP + KMeans on the RTX 3090). The default Python 3.14
kernel falls back to sklearn + umap-learn on CPU, which is also fine for
datasets up to ~50k rows.

**Parallelism**: set `WIKI_N_JOBS` in `.env` (default `-1` = all cores).

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

REPO = Path("../").resolve()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

from wiki_dumps.settings import get_settings

settings = get_settings()

DB_PATH    = REPO / "data/databases/scifi.db"  # ← change to any extract DB
TABLE      = "scifi_novels"                      # ← table with page_id column
N_JOBS     = settings.n_jobs                     # -1 = all cores (WIKI_N_JOBS in .env)
N_CLUSTERS = 7                                  # KMeans k
MAX_PAGES  = None                                # None = all; set int to prototype on subset
DUMP_DIR   = REPO / settings.dump_dir            # absolute path from repo root

print(f"DB:       {DB_PATH}")
print(f"Table:    {TABLE}")
print(f"n_jobs:   {N_JOBS}")
print(f"Dump dir: {DUMP_DIR}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ['CUDA_VISIBLE_DEVICES']}")

DB:       /home/benwright/Documents/GitHub/wiki/data/databases/scifi.db
Table:    scifi_novels
n_jobs:   -1
Dump dir: /home/benwright/Documents/GitHub/wiki/data/dumps
CUDA_VISIBLE_DEVICES: 1


In [124]:
# ── GPU detection ──────────────────────────────────────────────────────────
# Full GPU path requires the "Python 3.12 (RAPIDS GPU)" Jupyter kernel.
# CPU fallback works in any kernel but is slower for large datasets.
try:
    import cuml  # type: ignore[import-untyped]
    from cuml.cluster import KMeans
    from cuml.manifold import UMAP
    GPU = True
    print(f"✓ cuML {cuml.__version__} — UMAP + KMeans on GPU")
except ImportError:
    from sklearn.cluster import KMeans
    from umap import UMAP
    GPU = False
    print("⚠ cuML not found — falling back to CPU (sklearn + umap-learn)")
    print("  Switch to the 'Python 3.12 (RAPIDS GPU)' kernel for GPU acceleration.")

✓ cuML 26.04.000 — UMAP + KMeans on GPU


In [125]:
if GPU:
    import cupy as cp  # type: ignore[import-untyped]
    dev = cp.cuda.Device(0)
    props = cp.cuda.runtime.getDeviceProperties(0)
    name = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    vram_gb = props["totalGlobalMem"] / 1e9
    print(f"Active GPU: {name}  ({vram_gb:.0f} GB VRAM)")

Active GPU: NVIDIA GeForce RTX 3090  (25 GB VRAM)


In [126]:
import warnings

import bz2
import numpy as np
import pandas as pd
import plotly.express as px
import mwparserfromhell
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sqlalchemy import create_engine

from wiki_dumps.dumps.index import parse_index
from wiki_dumps.dumps.preprocess import load_raw_index, raw_paths
from wiki_dumps.parse.stream import iter_pages_from_xml_bytes

warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)

## 1  Load page IDs from extract DB

In [127]:
engine = create_engine(f"sqlite:///{DB_PATH}")
df = pd.read_sql_table(TABLE, engine)
engine.dispose()

if MAX_PAGES:
    df = df.head(MAX_PAGES)

print(f"Loaded {len(df):,} rows from {TABLE}")
display(df.head(3))

Loaded 6,306 rows from scifi_novels


,id,page_id,title,author,pub_year,publisher,series,genres,preceded_by,followed_by,description
0,1,2080,A Fire Upon the Deep,Vernor Vinge,1992.0,Tor Books,Zones of Thought series,Hard science fiction,None,A Deepness in the Sky,A Fire Upon the Deep is a 1992 science fiction...
1,2,4082,Blade Runner 2: The Edge of Human,K. W. Jeter,1995.0,None,Blade Runner,Science fiction,Do Androids Dream of Electric Sheep?,Replicant Night,Blade Runner 2: The Edge of Human (1995) is a ...
2,3,843,A Clockwork Orange (novel),Anthony Burgess,1962.0,Penguin Random House,None,Science fiction|dystopian fiction|satire|black...,None,None,A Clockwork Orange is a novel by the English w...


## 2  Build dump page-ID → block index

Maps every `page_id` to the raw-file block that contains it so we can seek directly
without scanning the whole dump.

In [128]:
DUMP_PATH  = sorted(DUMP_DIR.glob("*-pages-articles-multistream.xml.bz2"))[-1]
INDEX_PATH = Path(str(DUMP_PATH).replace(
    "-pages-articles-multistream.xml.bz2",
    "-pages-articles-multistream-index.txt.bz2",
))
RAW_PATH, IDX_PATH = raw_paths(DUMP_PATH)
assert RAW_PATH.exists(), f"Preprocessed raw file not found: {RAW_PATH}\nRun: uv run wiki dump preprocess"

raw_index = load_raw_index(IDX_PATH)

# Cache file lives next to the index; name includes mtime so it's
# automatically stale if the dump is ever re-downloaded.
_mtime    = int(INDEX_PATH.stat().st_mtime)
_cache    = INDEX_PATH.with_suffix(f".{_mtime}.pid_blk.npz")

if _cache.exists():
    _data   = np.load(_cache)
    pid_arr = _data["pid_arr"]
    blk_arr = _data["blk_arr"]
    print(f"Loaded pid/blk index from cache  ({_cache.name})")
else:
    # Clean up any stale caches for this dump before writing a new one
    for _old in INDEX_PATH.parent.glob(INDEX_PATH.stem + ".*.pid_blk.npz"):
        _old.unlink()

    _pids, _blks, block_idx, prev_offset = [], [], -1, -1
    with bz2.open(INDEX_PATH, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            fc = line.index(":")
            sc = line.index(":", fc + 1)
            offset  = int(line[:fc])
            page_id = int(line[fc + 1 : sc])
            if offset != prev_offset:
                block_idx += 1
                prev_offset = offset
            _pids.append(page_id)
            _blks.append(block_idx)

    pid_arr = np.array(_pids, dtype=np.int64)
    blk_arr = np.array(_blks, dtype=np.int32)
    order   = np.argsort(pid_arr, kind="stable")
    pid_arr = pid_arr[order]
    blk_arr = blk_arr[order]
    del _pids, _blks, order

    np.savez_compressed(_cache, pid_arr=pid_arr, blk_arr=blk_arr)
    print(f"Built and cached pid/blk index  ({_cache.name})")

print(f"Index: {len(pid_arr):,} pages in {blk_arr.max()+1:,} blocks")

Loaded pid/blk index from cache  (enwiki-20260301-pages-articles-multistream-index.txt.1777727529.pid_blk.npz)
Index: 25,432,678 pages in 254,363 blocks


## 3  Fetch + strip wikitext — process-parallel, block-batched

Groups page IDs by their raw block so each block is read once. Worker processes
bypass the GIL entirely (no shared state — each gets a plain tuple of ints).

In [129]:
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

page_ids = df["page_id"].tolist()

# Group page IDs by block so we read each block at most once
_block_to_pids: dict[int, list[int]] = defaultdict(list)
for pid in page_ids:
    pos = int(np.searchsorted(pid_arr, pid))
    if pos < len(pid_arr) and pid_arr[pos] == pid:
        _block_to_pids[int(blk_arr[pos])].append(pid)

print(f"{len(page_ids):,} pages span {len(_block_to_pids):,} unique blocks "
      f"(avg {len(page_ids)/max(len(_block_to_pids),1):.1f} pages/block)")

6,306 pages span 5,639 unique blocks (avg 1.1 pages/block)


In [130]:
def _fetch_block(args: tuple[int, int, list[int], str]) -> dict[int, str]:
    """Read one raw block and strip wikitext for all requested page IDs.

    Fully self-contained — no globals, safe for spawn-based multiprocessing.
    """
    import mwparserfromhell  # local import: each worker process needs its own
    from wiki_dumps.parse.stream import iter_pages_from_xml_bytes

    offset, length, pids, raw_path = args
    want = set(pids)
    results: dict[int, str] = {}
    with open(raw_path, "rb") as fh:
        fh.seek(offset)
        xml_bytes = fh.read(length)
    for page in iter_pages_from_xml_bytes(xml_bytes):
        if page.page_id in want:
            try:
                results[page.page_id] = mwparserfromhell.parse(page.wikitext).strip_code()
            except Exception:
                results[page.page_id] = page.wikitext
            if len(results) == len(want):
                break  # found everything in this block
    return results


# Build one arg-tuple per block (plain ints/str — cheap to pickle)
_raw_path_str = str(RAW_PATH)
_block_args = [
    (raw_index[blk][0], raw_index[blk][1], pids, _raw_path_str)
    for blk, pids in _block_to_pids.items()
]

n_workers = N_JOBS if N_JOBS > 0 else os.cpu_count()
print(f"Fetching {len(_block_args):,} blocks across {n_workers} processes …")

with ProcessPoolExecutor(max_workers=n_workers) as ex:
    _results = list(ex.map(_fetch_block, _block_args, chunksize=20))

# Merge all block results and reassemble in original page_ids order
pid_to_text: dict[int, str] = {}
for r in _results:
    pid_to_text.update(r)
corpus = [pid_to_text.get(pid, "") for pid in page_ids]

n_empty = sum(1 for t in corpus if not t)
print(f"Done — {len(corpus):,} docs | {n_empty:,} empty")

Fetching 5,639 blocks across 32 processes …
Done — 6,306 docs | 0 empty


## 4  TF-IDF vectorization

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_DOMAIN_STOPWORDS = {
    # Wikipedia boilerplate
    "redirect", "external", "links", "contents", "references", "isbn",
    "retrieved", "archived", "thumb", "right", "left", "edit",
    # Wikipedia category artifacts (show up after wikitext stripping)
    "category",
    # Wikipedia section headers
    "language", "reception", "plot", "synopsis",
    # Generic "based on the long-running television..." boilerplate
    "based", "running", "television", "long",
    # Generic fiction / article scaffolding
    "novel", "novels", "novella", "novelette", "novelettes","children","fantasy","young","adult","won","hugo","nebula",
    "anthologies", "anthology", "edited", "anthologies","title","director","released",
    "book", "books", "story", "stories", "series",
    "review", "reviews", "reviewed",
    "published", "publication", "author", "authors", "written", "writer", "writers",
    "magazine","issue","appeared","summary",
    "filming", "filmmaker", 
    "fiction", "science", "character", "characters",
    "set", "later", "new", "life", "people",
    "short","short short","story","short story","nominated",
    "film", "films", "american", "british", "english", "canadian",
    "award", "awards", "works", "work", "best", "introduction",
    "paperback", "hardcover", "edition", "volume", "page", "pages",
    "collection", "collections", "adventures",
    # Ubiquitous sci-fi nouns — appear in every cluster, discriminate nothing
    "earth", "human", "humans", "planet", "planets",
    "space", "time", "years", "year", "man", "men", "called", "world",
    # Publisher name fragments bleeding in as noise
    "del", "rey", "signet", "tor", "ace", "bantam", "baen",
}
all_stop_words = list(ENGLISH_STOP_WORDS | _DOMAIN_STOPWORDS)

vectorizer = TfidfVectorizer(
    max_features=2000,
    min_df=0.05,
    max_df=0.85,
    sublinear_tf=True,
    strip_accents="unicode",
    stop_words=all_stop_words,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b",
)
X = vectorizer.fit_transform(corpus)
print(f"TF-IDF matrix: {X.shape[0]:,} docs × {X.shape[1]:,} terms  ({X.nnz:,} non-zeros)")

TF-IDF matrix: 6,306 docs × 972 terms  (583,337 non-zeros)


## 5  Truncated SVD → dense embedding

Reduces the sparse TF-IDF matrix to 128 dense dimensions before UMAP.

In [132]:
SVD_COMPONENTS = 128
svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
X_svd = svd.fit_transform(X)
print(f"SVD variance explained: {svd.explained_variance_ratio_.sum():.1%}")

SVD variance explained: 63.1%


## 6  UMAP → 2-D embedding

In [133]:
umap_kwargs = dict(n_components=2, n_neighbors=15, min_dist=0.05, random_state=42, verbose=True)
if not GPU:
    umap_kwargs["n_jobs"] = N_JOBS

reducer = UMAP(**umap_kwargs)
embedding = reducer.fit_transform(X_svd)  # shape (n_docs, 2)
print(f"Embedding shape: {embedding.shape}")

[2026-05-04 22:18:06.359] [CUML] [debug] Building knn graph using build_algo='brute_force_knn'
[2026-05-04 22:18:06.388] [CUML] [debug] Computing KNN Graph
[2026-05-04 22:18:06.395] [CUML] [debug] Computing fuzzy simplicial set
Embedding shape: (6306, 2)


## 7  KMeans clustering

In [134]:
km_kwargs = dict(n_clusters=N_CLUSTERS, random_state=42)
if not GPU:
    km_kwargs["n_init"] = "auto"

km = KMeans(**km_kwargs)
# Cluster on SVD embedding (higher-dim = better separation than 2-D UMAP)
labels = km.fit_predict(X_svd)
if hasattr(labels, "to_numpy"):   # cuML returns cuDF series
    labels = labels.to_numpy()

df["cluster"] = labels
print("Cluster sizes:")
print(pd.Series(labels).value_counts().sort_index().to_string())

Cluster sizes:
0    1048
1    1186
2     510
3     635
4     548
5     971
6     136
7     245
8     296
9     731


## 8  Interactive scatter plot

In [143]:
emb = np.array(embedding)  # (n, 2) — ensure numpy from cuML

# ── per-cluster top-3 terms for centroid labels ─────────────────────────────
terms = np.array(vectorizer.get_feature_names_out())
cluster_labels_text: dict[int, str] = {}
for cid in range(N_CLUSTERS):
    mask = labels == cid
    if not mask.any():
        continue
    centroid_tfidf = np.asarray(X[mask].mean(axis=0)).ravel()
    top3 = terms[centroid_tfidf.argsort()[::-1][:3]]
    cluster_labels_text[cid] = "\n".join(top3)

# ── build plot dataframe ─────────────────────────────────────────────────────
plot_df = df[["title", "cluster"]].copy()
plot_df["x"] = emb[:, 0]
plot_df["y"] = emb[:, 1]
for col in ("pub_year", "release_year", "author", "director", "birth_year"):
    if col in df.columns:
        plot_df[col] = df[col]
        break
plot_df["cluster_str"] = plot_df["cluster"].astype(str)

# ── centroid positions ───────────────────────────────────────────────────────
centroids = (
    plot_df.groupby("cluster")[["x", "y"]].mean().reset_index()
)
centroids["label"] = centroids["cluster"].map(cluster_labels_text)
centroids["size"]  = plot_df.groupby("cluster").size().values

# ── scatter ──────────────────────────────────────────────────────────────────
hover_cols = [c for c in plot_df.columns if c not in ("x", "y", "cluster", "cluster_str")]
fig = px.scatter(
    plot_df, x="x", y="y",
    color="cluster_str",
    hover_data=hover_cols,
    title=f"TF-IDF clusters — {TABLE}  ({N_CLUSTERS} clusters · UMAP 2-D)",
    width=1150, height=750,
    labels={"color": "cluster"},
    opacity=0.55,
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(marker_size=4)

# Annotate each centroid with its top-3 terms + cluster id
"""
for _, row in centroids.iterrows():
    fig.add_annotation(
        x=row["x"], y=row["y"],
        text=f"<b>{int(row['cluster'])}</b><br><i>{row['label'].replace(chr(10), '<br>')}</i>",
        showarrow=False,
        font=dict(size=10, color="black"),
        bgcolor="rgba(255,255,255,0.75)",
        bordercolor="rgba(0,0,0,0.3)",
        borderwidth=1,
        borderpad=3,
    )
"""

fig.update_layout(legend_title_text="cluster")
fig.show()

In [136]:
TOP_N_TREEMAP = 8

treemap_rows = []
for cid in range(N_CLUSTERS):
    mask = labels == cid
    if not mask.any():
        continue
    centroid_tfidf = np.asarray(X[mask].mean(axis=0)).ravel()
    top_terms_str = " · ".join(terms[centroid_tfidf.argsort()[::-1][:TOP_N_TREEMAP]])
    treemap_rows.append({
        "cluster": f"Cluster {cid}",
        "size": int(mask.sum()),
        "top_terms": top_terms_str,
    })

tm_df = pd.DataFrame(treemap_rows)

fig2 = px.treemap(
    tm_df,
    path=["cluster"],
    values="size",
    custom_data=["top_terms", "size"],
    title=f"Cluster sizes — {TABLE}",
    color="size",
    color_continuous_scale="Blues",
    width=1150, height=500,
)
fig2.update_traces(
    texttemplate="<b>%{label}</b><br>%{customdata[1]} pages<br><br>%{customdata[0]}",
    textfont_size=12,
    hovertemplate="<b>%{label}</b><br>%{customdata[1]} pages<br>%{customdata[0]}<extra></extra>",
)
fig2.update_layout(coloraxis_showscale=False)
fig2.show()

## 9  Top TF-IDF terms per cluster

In [142]:
import scipy.sparse as sp

terms = np.array(vectorizer.get_feature_names_out())
TOP_N = 15

rows = []
for cluster_id in range(N_CLUSTERS):
    mask = labels == cluster_id
    if not mask.any():
        continue
    centroid = np.asarray(X[mask].mean(axis=0)).ravel()
    top_idx  = centroid.argsort()[::-1][:TOP_N]
    rows.append({
        "cluster": cluster_id,
        "size":    int(mask.sum()),
        "top_terms": ", ".join(terms[top_idx]),
    })

cluster_summary = pd.DataFrame(rows).sort_values(by='size',ascending=False).set_index("cluster")

with pd.option_context("display.max_colwidth", None):
    display(cluster_summary)

,size,top_terms
cluster,,
1,1186,"doctor, star, original, wars, features, cover, library, trilogy, second, universe, david, sequel, galaxy, moon, originally"
0,1048,"future, history, united, war, states, united states, century, writing, described, society, wrote, press, publishers, like, weekly"
5,971,"war, group, way, killed, use, like, help, city, escape, control, power, end, eventually, return, known"
9,731,"originally, asimov, astounding, locus, isaac asimov, isaac, tales, magazines, galaxy, horror, reprinted, copies, included, contains, house"
3,635,"father, mother, family, old, son, brother, house, daughter, finds, love, child, takes, girl, death, home"
4,548,"directed, production, adaptation, adapted, cast, release, movie, produced, original, shot, united, said, states, wrote, million"
2,510,"ship, crew, captain, star, alien, mission, ships, aboard, spaceship, universe, like, war, return, original, race"
8,296,"john, originally, worlds, astounding, david, press, described, april, locus, star, alien, james, december, future, miller"
7,245,"robert, john, james, press, martin, michael, asimov, galaxy, william, brian, jack, isaac, astounding, originally, charles"


## 10  Find similar pages

Enter a title and retrieve the most similar pages by cosine similarity in SVD space.

In [138]:
from sklearn.metrics.pairwise import cosine_similarity

QUERY_TITLE = df["title"].iloc[0]   # ← change to any title in the table
TOP_K = 10

idx = df.index[df["title"] == QUERY_TITLE]
assert len(idx) > 0, f"{QUERY_TITLE!r} not found in table"
query_vec = X_svd[idx[0]:idx[0]+1]

sims = cosine_similarity(query_vec, X_svd).ravel()
top_idx = sims.argsort()[::-1][1:TOP_K+1]  # skip self

result = df.iloc[top_idx][["title", "cluster"] + [c for c in ("pub_year", "release_year", "author", "director") if c in df.columns]].copy()
result["similarity"] = sims[top_idx].round(4)
print(f"Most similar to: {QUERY_TITLE!r}")
display(result)

Most similar to: 'A Fire Upon the Deep'


,title,cluster,pub_year,author,similarity
21,Ringworld,2,1970.0,Larry Niven,0.7050
278,Singularity Sky,5,2003.0,Charles Stross,0.6547
2645,House of Suns,5,2008.0,Alastair Reynolds,0.6384
4527,The Engines of God,2,1994.0,Jack McDevitt,0.6380
289,Perry Rhodan,5,NaN,Deborah Painter,0.6338
347,Shade's Children,5,1997.0,Garth Nix,0.6321
2751,Blindsight (Watts novel),2,2006.0,Peter Watts,0.6319
769,Gridlinked,2,2001.0,Neal Asher,0.6316
900,The War Against the Chtorr,5,1983.0,David Gerrold,0.6314
264,The Mote in God's Eye,2,1974.0,Larry NivenJerry Pournelle,0.6308
